# Airline Arrival Delay Analysis: Alaska Airlines vs. AM West

The objective of this analysis is to evaluate operational punctuality between Alaska Airlines and AM West across five shared metropolitan destinations: Los Angeles, Phoenix, San Diego, San Francisco, and Seattle. By examining arrival delays at both an aggregate level and a route-specific level, this study assesses overall carrier performance and identifies the structural factors governing arrival punctuality.

In [ ]:
# Step 1: Create flight_delays.csv file programmatically
csv_data = """Airline,Status,Los Angeles,Phoenix,San Diego,San Francisco,Seattle
ALASKA,on time,497,221,212,503,1841
ALASKA,delayed,62,12,20,102,305
AM WEST,on time,694,4840,383,320,201
AM WEST,delayed,117,415,65,129,61"""

with open("flight_delays.csv", "w") as f:
    f.write(csv_data)

print("Saved flight_delays.csv successfully.")

Saved flight_delays.csv successfully.


In [ ]:
import pandas as pd

# Load the raw dataset
df_raw = pd.read_csv("flight_delays.csv")
display(df_raw)

,Airline,Status,Los Angeles,Phoenix,San Diego,San Francisco,Seattle
0,ALASKA,on time,497,221,212,503,1841
1,ALASKA,delayed,62,12,20,102,305
2,AM WEST,on time,694,4840,383,320,201
3,AM WEST,delayed,117,415,65,129,61


In [ ]:
# Transform the wide table into a tidy long-form structure
df_tidy = pd.melt(
    df_raw,
    id_vars=["Airline", "Status"],
    var_name="Destination",
    value_name="Flight_Count"
)

display(df_tidy.head(10))

,Airline,Status,Destination,Flight_Count
0,ALASKA,on time,Los Angeles,497
1,ALASKA,delayed,Los Angeles,62
2,AM WEST,on time,Los Angeles,694
3,AM WEST,delayed,Los Angeles,117
4,ALASKA,on time,Phoenix,221
5,ALASKA,delayed,Phoenix,12
6,AM WEST,on time,Phoenix,4840
7,AM WEST,delayed,Phoenix,415
8,ALASKA,on time,San Diego,212
9,ALASKA,delayed,San Diego,20


In [ ]:
# Aggregate total arrivals by airline and arrival status
airline_totals = df_tidy.pivot_table(
    index="Airline",
    columns="Status",
    values="Flight_Count",
    aggfunc="sum"
)

# Calculate flight volumes and gross delay percentages
airline_totals["Total_Flights"] = airline_totals["delayed"] + airline_totals["on time"]
airline_totals["Delay_Rate_%"] = (airline_totals["delayed"] / airline_totals["Total_Flights"]) * 100

display(round(airline_totals, 2))

Status,delayed,on time,Total_Flights,Delay_Rate_%
Airline,,,,
ALASKA,501,3274,3775,13.27
AM WEST,787,6438,7225,10.89


### Initial Aggregate Assessment

At first glance, top-level consolidated figures suggest that AM West maintains superior operational reliability. Across all five shared destinations, AM West registers an overall delay rate of 10.89% across 7,225 total flights, whereas Alaska Airlines records a higher delay rate of 13.27% across 3,775 total flights. Evaluating carrier performance strictly through high-level aggregates would lead to the conclusion that AM West delivers stronger on-time execution.

In [ ]:
# Disaggregate arrivals by destination and carrier
city_breakdown = df_tidy.pivot_table(
    index=["Destination", "Airline"],
    columns="Status",
    values="Flight_Count"
).reset_index()

city_breakdown["Total_Flights"] = city_breakdown["delayed"] + city_breakdown["on time"]
city_breakdown["Delay_Rate_%"] = (city_breakdown["delayed"] / city_breakdown["Total_Flights"]) * 100

# Format side-by-side comparison
city_comparison = city_breakdown.pivot(
    index="Destination",
    columns="Airline",
    values="Delay_Rate_%"
)

city_comparison["Alaska_Advantage_%"] = city_comparison["AM WEST"] - city_comparison["ALASKA"]
display(round(city_comparison, 2))

Airline,ALASKA,AM WEST,Alaska_Advantage_%
Destination,,,
Los Angeles,11.09,14.43,3.34
Phoenix,5.15,7.90,2.75
San Diego,8.62,14.51,5.89
San Francisco,16.86,28.73,11.87
Seattle,14.21,23.28,9.07


In [ ]:
# Assess geographic flight allocation per airline
flight_distribution = df_tidy.groupby(["Destination", "Airline"])["Flight_Count"].sum().unstack()
flight_distribution["ALASKA_Share_%"] = (flight_distribution["ALASKA"] / flight_distribution["ALASKA"].sum()) * 100
flight_distribution["AM_WEST_Share_%"] = (flight_distribution["AM WEST"] / flight_distribution["AM WEST"].sum()) * 100

display(round(flight_distribution, 2))

Airline,ALASKA,AM WEST,ALASKA_Share_%,AM_WEST_Share_%
Destination,,,,
Los Angeles,559,811,14.81,11.22
Phoenix,233,5255,6.17,72.73
San Diego,232,448,6.15,6.20
San Francisco,605,449,16.03,6.21
Seattle,2146,262,56.85,3.63


## Analysis and Conclusions

### 1. Route-by-Route Performance vs. Aggregate (Simpson's Paradox)
When we look at individual cities, **Alaska Airlines consistently has a lower delay rate than AM West on every single route**:

* **Los Angeles:** Alaska (11.09%) is better than AM West (14.43%)
* **Phoenix:** Alaska (5.15%) is better than AM West (7.90%)
* **San Diego:** Alaska (8.62%) is better than AM West (14.51%)
* **San Francisco:** Alaska (16.86%) is better than AM West (28.73%)
* **Seattle:** Alaska (14.21%) is better than AM West (23.28%)

Despite this, AM West has a lower **total** delay rate (10.89%) than Alaska (13.27%). This is a classic example of **Simpson's Paradox**, where a trend appearing in subgroups reverses when the subgroups are combined.

### 2. Why Does This Paradox Happen?
The contradiction is caused by where each airline flies most of its planes:

* **AM West** routes **72.7%** of its flights through **Phoenix**, a low-delay airport (overall delay rates under 8%). This highly weights AM West's overall performance toward on-time arrivals.
* **Alaska** routes **56.8%** of its flights through **Seattle**, an airport prone to bad weather and high baseline delays (overall delay rates up to 23%). This drags down Alaska's overall aggregate score.

### 3. Final Conclusion
Aggregate numbers are misleading here. If you choose between the two carriers on any shared route, you are statistically less likely to experience a delay if you fly with **Alaska Airlines**.